# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1. Unit of Analysis:** One row = **one pseudonymized content item (page) for a given client** over a trailing 90-day aggregation window.

**2. Time Window:** For our mid-panel validation month, we analyze **March 2026** (`month=2026-03`):
- **Feature Window:** December 1, 2025 to February 28, 2026 (90-day historical window).
- **Target Window:** March 1, 2026 to March 31, 2026 (31-day future window).
- **Active Slice Filter:** We filter on pages with at least 100 GSC impressions in February 2026 (`imp_feb >= 100`) to focus on active organic opportunities.

In [1]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from google.colab import userdata

# Set Hugging Face Token
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_src = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
march_src = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**1. Features (Maximum 5):**
- `imp_prev90`: GSC impressions in the trailing 90 days. *Available when?* Knowable before the prediction moment (March 1, 2026) because it uses historical measurements from the past 90 days.
- `clk_prev90`: GSC clicks in the trailing 90 days. *Available when?* Knowable before the prediction moment because clicks are recorded daily alongside impressions in historical search console data.
- `pos_prev30`: GSC average position in the trailing 30 days of the feature window (Feb 1, 2026 - Feb 28, 2026). *Available when?* Knowable before the prediction moment as it calculates the page's average ranking position in the month leading up to March 1.
- `sessions_prev90`: GA4 page sessions in the trailing 90 days. *Available when?* Knowable before the prediction moment as it aggregates user analytics sessions over the past 90 days.
- `imp_feb`: GSC impressions in February 2026 (Feb 1 - Feb 28, 2026). *Available when?* Knowable before the prediction moment as it is the final month's baseline volume before March 1.

**2. Label / Proxy:**
- `is_declining`: Defined as `1` if `imp_march < 0.8 * imp_feb`, else `0` (where `imp_march` is total GSC impressions in March 2026).

**3. Context Fields:**
- `content_hash_id`: Stable pseudonymous page identifier. Join/group only.
- `client_hash_id`: Stable pseudonymous client identifier. Client-holdout splits only.

**4. Excluded Fields:**
- `health_score`, `priority_score`, `action_type`: Excluded because they are downstream outputs of the existing rule-based system. Using them would cause circular logic.
- `imp_last30` / `clk_last30` / `imp_march` (when used as features): Excluded because they overlap with the target prediction period (March 2026) and introduce future data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We run three verification queries on the mid-panel month (`month=2026-03`) to prove our contract claims:
1. **Query 1: Grain verification** - Ensure there are no duplicate records at the `(report_date, client_hash_id, content_hash_id)` grain.
2. **Query 2: Slice counts and Date span** - Check row counts and min/max date coverage of the March 2026 slice.
3. **Query 3: Availability (ga4_data_available IS TRUE)** - Count rows where GA4 analytics data is active.

We then build a small 5-feature frame with the target column and perform the leakage experiment.

In [2]:
# Query 1: Grain verification
print("=== Query 1: Grain verification ===")
grain_df = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
    FROM {march_src}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate count: {len(grain_df)}")

# Query 2: Slice counts and Date span
print("\n=== Query 2: Slice counts and Date span ===")
slice_df = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM {march_src}
""").df()
print(slice_df.to_string(index=False))

# Query 3: Availability (ga4_data_available IS TRUE)
print("\n=== Query 3: Availability ===")
availability_df = con.sql(f"""
    SELECT COUNT(*) as active_ga4_rows
    FROM {march_src}
    WHERE ga4_data_available IS TRUE
""").df()
print(availability_df.to_string(index=False))

=== Query 1: Grain verification ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate count: 0

=== Query 2: Slice counts and Date span ===
 total_rows   min_date   max_date
    9841378 2026-03-01 2026-03-31

=== Query 3: Availability ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 active_ga4_rows
          413966


In [3]:
# Build small feature frame
print("=== Building Feature Frame (March 2026) ===")
data = con.sql(f"""
    WITH features_agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date BETWEEN '2025-12-01' AND '2026-02-28' THEN gsc_impressions ELSE 0 END) AS imp_prev90,
               SUM(CASE WHEN report_date BETWEEN '2025-12-01' AND '2026-02-28' THEN gsc_clicks ELSE 0 END) AS clk_prev90,
               AVG(CASE WHEN report_date BETWEEN '2026-02-01' AND '2026-02-28' THEN gsc_avg_position END) AS pos_prev30,
               SUM(CASE WHEN report_date BETWEEN '2025-12-01' AND '2026-02-28' THEN ga4_sessions ELSE 0 END) AS sessions_prev90,
               SUM(CASE WHEN report_date BETWEEN '2026-02-01' AND '2026-02-28' THEN gsc_impressions ELSE 0 END) AS imp_feb
        FROM {fact_src}
        WHERE report_date BETWEEN '2025-12-01' AND '2026-02-28'
        GROUP BY 1, 2
        HAVING imp_feb >= 100
    ),
    label_agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_march
        FROM {fact_src}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT f.*, COALESCE(l.imp_march, 0) AS imp_march
    FROM features_agg f
    LEFT JOIN label_agg l ON f.client_hash_id = l.client_hash_id AND f.content_hash_id = l.content_hash_id
""").df()

data['is_declining'] = (data['imp_march'] < 0.8 * data['imp_feb']).astype(int)
data['pos_prev30'] = data['pos_prev30'].fillna(0)
data['sessions_prev90'] = data['sessions_prev90'].fillna(0)

print(f"Feature frame shape: {data.shape}")
print(data[['content_id' if 'content_id' in data.columns else 'content_hash_id', 'imp_prev90', 'clk_prev90', 'pos_prev30', 'sessions_prev90', 'imp_feb', 'is_declining']].head())

=== Building Feature Frame (March 2026) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (80322, 9)
            content_hash_id  imp_prev90  clk_prev90  pos_prev30  \
0  content_a9a7ac6e507f22bb      2874.0         9.0    5.764576   
1  content_d30d7688a5a5580d       869.0         0.0   12.625969   
2  content_675d089961ded163      1973.0         1.0    3.343598   
3  content_b407c423d6ece6cb      5590.0         1.0    3.886806   
4  content_498a9afb9da07372      3276.0         1.0    3.621873   

   sessions_prev90  imp_feb  is_declining  
0              7.0   1538.0             0  
1              0.0    491.0             0  
2              4.0    790.0             0  
3              4.0   1868.0             0  
4              5.0   1361.0             0  


In [4]:
# The Leakage Experiment (Trap)
feature_cols = ['imp_prev90', 'clk_prev90', 'pos_prev30', 'sessions_prev90', 'imp_feb']

X = data[feature_cols]
y = data['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print("=== Trapping ourselves: training Leaky Model ===")
X_leaky = X.copy()
X_leaky['imp_march_leaky'] = data['imp_march'] # Direct target leakage
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
model_leaky = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr_l)
print(f"Leaky ROC AUC: {roc_auc_score(y_te_l, model_leaky.predict_proba(X_te_l)[:, 1]):.3f}")

print("\n=== Deleting the leak: training Honest Model ===")
model_honest = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print(f"Honest ROC AUC: {roc_auc_score(y_te, model_honest.predict_proba(X_te)[:, 1]):.3f}")

=== Trapping ourselves: training Leaky Model ===
Leaky ROC AUC: 0.999

=== Deleting the leak: training Honest Model ===
Honest ROC AUC: 0.651


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Key Limitations of this slice:**
1. **Unbalanced Panel History Depth:** The client enrollment is highly staggered. Some clients have history dating back to early 2025, while others only start in late 2025 or early 2026. Performing a global time aggregation means some clients will have high numbers of missing values at the beginning of the timeline.
2. **GA4 Data Sparsity:** While Google Search Console (GSC) is available for most historical rows, GA4 analytics availability is extremely thin. In March 2026, only **4.2%** of the rows (413,966 out of 9,841,378 rows) have GA4 tracking available (`ga4_data_available IS TRUE`). For the other ~95% of rows, the GA4 columns are zero-filled. If models are trained blindly on raw GA4 numbers without checking the availability flag, they will mistake "tracking not set up" for "zero engagement."
3. **Missing Keyword Context:** Keyword volume and CPC features are missing systemically by `content_type` (e.g. comparison and feedly articles do not record keywords), which limits multivariate keyword analysis on certain page segments.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.